# Renewable-aware 24-hour planning

A demand forecast says how much electricity customers may use, but it does not say how much of that demand may coincide with solar and wind availability. This notebook explains the transparent planning layer built from the saved **Recursive XGBoost** demand forecasts. It supports human review; it is not an automatic grid controller.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results' / 'planning').exists():
    ROOT = ROOT.parent
TABLES = ROOT / 'results' / 'planning' / 'tables'
planning = pd.read_csv(TABLES / 'renewable_planning_predictions.csv', parse_dates=['forecast_origin', 'forecast_date', 'target_timestamp'])
method_metrics = pd.read_csv(TABLES / 'renewable_method_validation_metrics.csv')
selected = pd.read_csv(TABLES / 'selected_renewable_method.csv').iloc[0]
thresholds = pd.read_csv(TABLES / 'planning_thresholds.csv')
daily = pd.read_csv(TABLES / 'daily_planning_summary.csv', parse_dates=['forecast_date'])
print(f"Planning rows: {len(planning):,}; selected renewable method: {selected['selected_method_label']}")

## Avoiding future leakage

At each forecast origin, renewable estimates may use only information already reported. The daily method reads 24 hours back, the weekly method reads 168 hours back, and the rolling method uses complete observations for the same UTC hour inside `(origin − 28 days, origin]`. Target-day renewable actuals are retained for retrospective scoring but never used to construct a prediction. Missing values are not filled, and negative reported solar values are preserved.

In [ ]:
comparison = method_metrics.loc[method_metrics['resource'].eq('combined_solar_wind'), ['method_label', 'count', 'mae_mwh', 'rmse_mwh', 'bias_mwh']].copy()
comparison = comparison.sort_values('mae_mwh')
comparison.round(2)

## Why validation chooses the method

The table above is intentionally limited to validation data. The method with the lowest validation combined-renewable MAE is frozen first; only then is test performance calculated. This keeps the test period as an honest final check instead of quietly tuning to it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(comparison['method_label'], comparison['mae_mwh'], color=['#4C78A8', '#54A24B', '#F58518'])
ax.set(title='Validation combined-renewable MAE', ylabel='MAE (MWh)', xlabel='Past-only method')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=.25)
plt.tight_layout()
plt.show()

## Residual demand and empirical scenarios

Residual demand is `forecast demand − forecast solar − forecast wind`. Only solar and wind are subtracted because those are the renewable measurements in this project. The result is **not** complete physical grid balance: other generation, imports, exports, storage, losses, reserves, and network constraints are absent.

For each target UTC hour, the planning scenarios use the past-only same-hour sample: P25 is conservative renewable availability, P50 is typical, and P75 is favourable. They are empirical scenarios, not statistically calibrated prediction intervals. Lower renewable availability produces higher residual demand.

In [ ]:
test_dates = daily.loc[daily['split'].eq('test'), 'forecast_date'].sort_values().reset_index(drop=True)
representative_date = test_dates.iloc[len(test_dates) // 2]
day = planning.loc[planning['forecast_date'].eq(representative_date)].sort_values('horizon')
fig, ax = plt.subplots(figsize=(10, 5))
for column, label in [
    ('conservative_residual_demand_scenario_mwh', 'Conservative renewable (P25)'),
    ('typical_residual_demand_scenario_mwh', 'Typical renewable (P50)'),
    ('favourable_residual_demand_scenario_mwh', 'Favourable renewable (P75)'),
]:
    ax.plot(day['target_timestamp'], day[column], label=label)
ax.set(title=f'Residual-demand scenarios: {representative_date:%Y-%m-%d} UTC', ylabel='Residual demand (MWh)', xlabel='Target time (UTC)')
ax.legend(); ax.grid(alpha=.25); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## Peaks, ramps, and training-only indicators

The thresholds below were fitted only on the 2022–2023 training period. A high-demand, high-residual, high-upward-ramp, or low-renewable-share flag highlights an hour for review. These are descriptive planning indicators—not safety guarantees, dispatch commands, or evidence of operating cost or savings. The first forecast horizon has no within-forecast ramp because there is no earlier forecast hour in that 24-hour block.

In [ ]:
thresholds[['threshold_name', 'threshold_value', 'quantile', 'fit_split', 'training_row_count']].round(3)

In [ ]:
alert_columns = [
    'hours_above_high_demand_threshold',
    'hours_above_high_residual_threshold',
    'hours_with_high_upward_ramp_alert',
    'hours_with_low_renewable_share_alert',
]
daily.groupby('split')[alert_columns].mean().round(2)

## Human planning, not automatic control

A dashboard can show the hourly master table, daily peaks, scenario bands, data-completeness flags, and error views by horizon, UTC hour, and month. A human planner can use those views to focus attention and compare plausible renewable conditions. The outputs should not be translated into automatic control actions because the analysis has no full supply stack, network model, reserve requirement, price data, or operational constraints.